In [ ]:
import numpy as np
import pandas as pd
from Influence_Modelling import XsvNetwork, Network, Sampling, Heuristics
p2p = XsvNetwork.XsvNetwork("p2p")
net=  Network.Network("p2p")
net.set_nx(p2p.get_nx())
import networkx as nx
import random

In [ ]:
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science', 'no-latex'])

def plot_degree_histogram(G, axis, methods, labels, show_baseline, proportion=0.1):
  assert len(methods) == len(labels)
  assert type(show_baseline) == bool
  if show_baseline:
    methods.append(Heuristics.random_h)
    labels.append("Random (Baseline)")
  for i, method in enumerate(methods):
    axis.hist([G.degree(g) for g in method(G, round(proportion*len(G.nodes())))], label=labels[i], alpha=0.5)
  axis.set_xlabel("Degree")
  axis.set_ylabel("Number of Nodes")
  axis.set_title("Degree Distribution")
  axis.legend()

In [ ]:
fig, ax = plt.subplots()
fig.set_dpi(220)
plot_degree_histogram(net.get_nx(), ax, [Heuristics.random_h], ["Baseline"], False, 1)

In [ ]:
ninesixtyp2p = Sampling.mhda_sample(net.get_nx(), 960)
fig, ax = plt.subplots()
fig.set_dpi(220)
plot_degree_histogram(ninesixtyp2p, ax, [Heuristics.random_h], ["Baseline"], False, 1)

In [ ]:
def navigable_small_world_graph(n: int):
  # generate a navigable small world graph to start as a baseline
  net = Network.Network("")
  net.kleinbergs_generator(n)
  return net.get_nx()

nsw = navigable_small_world_graph(1000)

In [ ]:
fig, ax = plt.subplots()
fig.set_dpi(220)
plot_degree_histogram(nsw, ax, [Heuristics.random_h], ["Baseline"], False, 1)

In [ ]:
def gnutella_dupe(n: int, p_long_connection=0.05, clust=0.4, edges_per_node=2):
  G = nx.powerlaw_cluster_graph(n, edges_per_node, clust)
  for v in list(G.nodes()):
    if random.random() <= 0.05:
      u = random.choice(list(G.nodes()))
      G.add_edge(u, v)
  for v in list(G.nodes()):
    if G.degree(v) > 15:
      edges = list(G.edges(v))
      for e in edges:
        G.remove_edge(*e)
  print(len(list(G.nodes())))
  return G

generated = gnutella_dupe(1000)
sampled = Sampling.mhda_sample(net.get_nx(), 1000)
print(len(list(sampled.nodes())))

fig, ax = plt.subplots()
fig.set_dpi(220)
plot_degree_histogram(generated, ax, [Heuristics.random_h], ['Generated'], False, 1)
plot_degree_histogram(sampled, ax, [Heuristics.random_h], ['Sampled from network'], False, 1)

In [ ]:
def plot_networks_against_method(title, ax, nets, net_labels, method, seeds=[x for x in range(1, 200, 5)]):
  assert type(title) == str
  assert len(net_labels) == len(nets) 
  assert ax is not None
  assert str(type(method)).split()[-1][1:-2] == "function"
  markers= ["s", "o", "^", "v"]
  ax.set_xlabel("Number of seed Nodes")
  ax.set_ylabel("Final Influence")
  data = []
  for i, net in enumerate(nets):
    n = Network.Network("")
    n.set_nx(nets[i])
    net_res =  [n.run_ICC(seed_nodes=Heuristics.random_h(nets[i], seed)) for seed in seeds]
    net_res = [de[2][-1] for de in net_res]
    data.append(net_res)
  for i, row in enumerate(data):
    ax.plot(seeds, row, label=net_labels[i], marker=markers[i % len(markers)], linewidth=0.5, markersize=2)
  ax.set_title(title)
  ax.legend()
  
fig, ax = plt.subplots()
fig.set_dpi(220)
plot_networks_against_method("Random Seed Nodes - 1000 Node Networks", ax, [generated, sampled, nsw], ["Generated", "Sampled p2p", "Kleinberg"], Heuristics.double_deg_discount_h)

In [ ]:
from Influence_Modelling import Metrics

In [ ]:
Metrics.generate_tabular_row(generated)

In [ ]:
Metrics.generate_tabular_row(sampled)

In [ ]:
Metrics.generate_tabular_row(nsw)